In [2]:
!pip install mlflow boto3 dagshub

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os

os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = 'NTsundere'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'your_key'

import mlflow
import numpy as np
import pandas as pd

In [5]:
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

2026/07/24 20:57:29 INFO mlflow.tracking.fluent: Experiment with name 'Exp 2 - BoW vs TfIdf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/829717980d5141c0b0549395ff0825ba', creation_time=1784915849318, experiment_id='2', last_update_time=1784915849318, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={}>

In [7]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import seaborn as sns

In [8]:
df = pd.read_csv("reddit_preprocessing.csv").dropna(subset='clean_comment')
df.shape

(36662, 2)

In [ ]:
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    with mlflow.start_run() as run:
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

ngram_ranges = [(1, 1), (1, 2), (1, 3)] 
max_features = 5000  

for ngram_range in ngram_ranges:
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")


2026/07/24 21:19:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 21:20:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run BoW_(1, 1)_RandomForest at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2/runs/50cd9d2524324c30adbed304b4fb5add
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2


2026/07/24 21:21:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 21:21:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 1)_RandomForest at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2/runs/cab4ba2e0bc84f85be547fd16de3e0d5
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2


2026/07/24 21:23:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 21:23:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2/runs/067b76aff7e14d94a5e4e4f83f65cbde
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2


2026/07/24 21:26:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 21:26:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run BoW_(1, 3)_RandomForest at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2/runs/b2f61a379df6452083d22d53eee1b08c
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2


2026/07/24 21:27:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/24 21:27:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TF-IDF_(1, 3)_RandomForest at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2/runs/8516cf76c0a34b3280494c1cedc5467b
🧪 View experiment at: https://dagshub.com/NTsundere/mlflow-reddit-sentiment.mlflow/#/experiments/2
